# Expected information gain and the value of information

One scalar parameter `θ` with a Gaussian prior of sd `prior_sd`, measured by an experiment
with standard error `experiment_se`. The expected information gain is the mutual information

    EIG = 0.5 · ln(1 + prior_sd² / experiment_se²)   nats,

and `eig_monte_carlo` is the nested Monte-Carlo estimator of the same quantity for a prior
given as draws. Information ages: the posterior *variance* doubles every half-life, so
`decayed_sd(sd, t, hl) = sd · 2^(t / 2hl)`.

In [ ]:
import numpy as np

from axiom.core import Unsupported
from axiom.design import (
    DESIGN_RELATIVE_SE, DecisionSpec, EIGEstimate, EVOIResult, ReExperimentTiming, ValueFn,
    decayed_sd, eig_gaussian, eig_monte_carlo, evoi_gaussian, evpi, evpi_gaussian, evsi,
    evsi_gaussian, experiment_se_for_design, information_half_life, preposterior_sd,
    preposterior_sd_ratio, time_to_re_experiment,
)

from axiom.display import enable

enable();  # every axiom result renders itself from here on

In [ ]:
print("relative SE by design kind:", dict(DESIGN_RELATIVE_SE))
se_holdout = experiment_se_for_design("cluster_holdout", reference_value=2.0)
se_ghost = experiment_se_for_design("ghost", reference_value=2.0)
print("cluster holdout se:", se_holdout, "| ghost se:", se_ghost)
print("unknown kind ->", type(experiment_se_for_design("crossover", 2.0)).__name__)
print("with explicit relative_se:", experiment_se_for_design("crossover", 2.0, relative_se=0.15))

In [ ]:
prior_sd = 0.5
for se in (se_holdout, se_ghost, 1.0):
    print(f"experiment_se={se:.2f}  EIG = {eig_gaussian(prior_sd, se):.4f} nats")

## Monte Carlo converges to the closed form

For a Gaussian prior the nested estimator agrees with `eig_gaussian` to within its own
Monte-Carlo standard error; the bias is `O(1 / n_prior_draws)`.

In [ ]:
rng = np.random.default_rng(0)
prior_draws = rng.normal(0.0, prior_sd, size=4000)
closed = eig_gaussian(prior_sd, 0.3)
for n_sims in (200, 1000, 4000):
    est: EIGEstimate = eig_monte_carlo(prior_draws, 0.3, n_sims=n_sims, seed=1)
    print(f"n_sims={n_sims:5d}  eig={est.eig:.4f} ± {est.se:.4f}   closed form {closed:.4f}")

In [ ]:
post_sd = 0.12
print("decayed sd after 0, 6, 12 periods (hl=6):", [round(decayed_sd(post_sd, t, 6.0), 4) for t in (0, 6, 12)])
print("information half-life:", round(information_half_life(prior_sd, post_sd, 6.0), 2), "periods")
timing: ReExperimentTiming = time_to_re_experiment(post_sd, 6.0, experiment_se=0.2, min_eig=0.3, design_kind="cluster_holdout")
print(f"EIG now {timing.eig_now:.3f}; worth {timing.min_eig} nats again after {timing.periods:.1f} periods (sd {timing.sd_at_threshold:.3f})")

## Value of information for a two-action decision

`DecisionSpec`: *act* pays `value_per_outcome_unit · (θ − threshold)`, *hold* pays nothing.
EVPI is what a clairvoyant would gain over acting on the prior mean alone,
`v · (E[max(θ − c, 0)] − max(Eθ − c, 0))`. EVSI replaces the prior sd with the preposterior sd
of the mean the experiment will produce, `prior_sd · sqrt(prior_sd² / (prior_sd² + se²))`.

In [ ]:
decision = DecisionSpec(name="scale_up", threshold=0.0, value_per_outcome_unit=1000.0, numeraire="USD")
mean, sd, se = 0.1, 0.5, 0.3
print("preposterior ratio:", round(preposterior_sd_ratio(sd, se), 4), "| preposterior sd:", round(preposterior_sd(sd, se), 4))
print("EVPI:", round(evpi_gaussian(decision, mean, sd), 2), decision.numeraire)
print("EVSI:", round(evsi_gaussian(decision, mean, sd, se), 2), decision.numeraire)
res: EVOIResult = evoi_gaussian(decision, mean, sd, se)
print(res.model_dump(include={"evpi", "evsi", "preposterior_sd", "method", "numeraire"}))

In [ ]:
draws = rng.normal(mean, sd, size=20000)
print("MC EVPI:", round(evpi(decision, draws), 2), "vs closed", round(evpi_gaussian(decision, mean, sd), 2))
for n_sims in (500, 4000):
    mc = evsi(decision, draws, se, n_sims=n_sims, seed=2)
    print(f"MC EVSI n_sims={n_sims}: {mc.evsi:.2f} ± {mc.evsi_se:.2f}   closed {res.evsi:.2f}")

In [ ]:
# A non-linear payoff of acting: capped upside.
capped: ValueFn = lambda theta: np.minimum(decision.payoff(theta), 200.0)
print("EVPI with capped payoff:", round(evpi(decision, draws, value_fn=capped), 2))
print("EVSI with capped payoff:", round(evsi(decision, draws, se, n_sims=2000, seed=3, value_fn=capped).evsi, 2))